In [ ]:
Practical 7: SQL operations using SQLite (built into Python)
import sqlite3
import pandas as pd
 
conn = sqlite3.connect(":memory:")       # temporary database in memory
cur = conn.cursor()
 
# ---------- 1. Create tables ----------
cur.execute("""CREATE TABLE department (
                   dept_id INTEGER PRIMARY KEY,
                   dept_name TEXT)""")
cur.execute("""CREATE TABLE employee (
                   emp_id INTEGER PRIMARY KEY,
                   name TEXT,
                   dept_id INTEGER,
                   salary INTEGER,
                   city TEXT)""")
 
# ---------- 2. Insert records ----------
cur.executemany("INSERT INTO department VALUES (?, ?)",
                [(1, "Sales"), (2, "IT"), (3, "HR"), (4, "Finance")])
cur.executemany("INSERT INTO employee VALUES (?, ?, ?, ?, ?)", [
    (101, "Amit", 2, 60000, "Pune"),
    (102, "Riya", 1, 45000, "Mumbai"),
    (103, "Karan", 2, 75000, "Pune"),
    (104, "Sneha", 3, 40000, "Nashik"),
    (105, "Rahul", 1, 52000, "Pune"),
    (106, "Pooja", 2, 68000, "Mumbai"),
    (107, "Vikas", 3, 38000, "Pune"),
    (108, "Neha", 1, 50000, "Nagpur"),
])
conn.commit()
 
def run(title, query):
    print("\n" + title)
    print(pd.read_sql_query(query, conn).to_string(index=False))
 
# ---------- 3. Queries ----------
run("1. SELECT - all employees",
    "SELECT * FROM employee")
 
run("2. FILTER - salary greater than 50000",
    "SELECT name, salary FROM employee WHERE salary > 50000")
 
run("3. FILTER with AND - city is Pune and salary >= 50000",
    """SELECT name, city, salary FROM employee
       WHERE city = 'Pune' AND salary >= 50000""")
 
run("4. SORT - highest salary first",
    "SELECT name, salary FROM employee ORDER BY salary DESC")
 
run("5. GROUP BY and aggregation - department wise summary",
    """SELECT dept_id, COUNT(*) AS total_emp,
              ROUND(AVG(salary), 2) AS avg_salary,
              MAX(salary) AS max_salary
       FROM employee GROUP BY dept_id""")
 
run("6. HAVING - departments with average salary above 50000",
    """SELECT dept_id, ROUND(AVG(salary), 2) AS avg_salary FROM employee
       GROUP BY dept_id HAVING AVG(salary) > 50000""")
 
run("7. INNER JOIN - employee with department name",
    """SELECT e.name, d.dept_name, e.salary
       FROM employee e INNER JOIN department d ON e.dept_id = d.dept_id
       ORDER BY d.dept_name""")
 
run("8. LEFT JOIN + GROUP BY - employees in every department",
    """SELECT d.dept_name, COUNT(e.emp_id) AS employees,
              COALESCE(SUM(e.salary), 0) AS total_salary
       FROM department d LEFT JOIN employee e ON d.dept_id = e.dept_id
       GROUP BY d.dept_name""")